# YouTube Recommendation Algorithm: A/B Experiment Analysis

**Author:** Praneeth Korukonda
**Date:** January 2024
**Role context:** Data Scientist, Product Analytics

---

> **Dataset note:** The dataset used in this analysis is synthetically generated
> using a realistic user-engagement model (log-normal watch times, latent user
> heterogeneity, a churn process, and calibrated session probabilities).
> All analytical decisions, experiment design, KPI selection, statistical
> methodology, root cause framing, and business interpretation, are original work.

---

## Business Question

YouTube is testing a new recommendation algorithm that surfaces more
**long-form educational content** in users' feeds.

**Central question:** Does the new algorithm increase long-term user engagement
without harming the creator ecosystem (particularly short-form / entertainment creators)?

## Experiment Design

| Parameter | Value |
|---|---|
| Experiment window | 30 days (Jan 15 – Feb 13, 2024) |
| Randomization | User-level, 50/50 split |
| Population | Active YouTube users (≥1 session in 30 days pre-experiment) |
| Treatment | New recommendation algorithm (more long-form educational content) |
| Control | Existing algorithm (default content mix) |

## Hypotheses

- **H₀:** New algorithm has no effect on avg daily watch time per user
- **H₁:** New algorithm increases avg daily watch time per user
- **α = 0.05**, two-tailed Welch's t-test


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, json
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 110, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'axes.titleweight': 'bold', 'figure.facecolor': 'white',
})
BLUE, RED, GREEN, AMBER = '#4285F4', '#EA4335', '#34A853', '#F9AB00'
print("Libraries loaded.")


---
## Section 1: Data Generation

The synthetic dataset models a realistic active-user cohort with three key properties:

1. **Latent engagement heterogeneity**: each user has a hidden engagement score that
   drives both their pre-experiment and in-experiment behavior, creating a meaningful
   covariate for CUPED.
2. **Log-normal session watch times**: right-skewed, matching the distribution
   observed in real streaming data (most sessions are short; a few are very long).
3. **Churn process**: ~45% of control and ~38% of treatment users trail off after
   ~2 weeks, producing realistic week-4 retention rates in the 70–78% range and a
   measurable retention gap between variants.


In [ ]:
N_USERS, N_DAYS = 5_000, 30
EXP_START = pd.Timestamp('2024-01-15')
rng = np.random.default_rng(42)

# Latent engagement score: shared driver for pre- and in-experiment behaviour
engagement = rng.normal(0, 1, N_USERS)

users = pd.DataFrame({
    'user_id'     : np.arange(1, N_USERS + 1),
    'variant'     : rng.choice(['control','treatment'], N_USERS, p=[.50,.50]),
    'age_group'   : rng.choice(['18-24','25-34','35-44','45-54','55+'], N_USERS,
                                p=[.25,.30,.22,.13,.10]),
    'country'     : rng.choice(['US','UK','India','Brazil','Canada'], N_USERS,
                                p=[.45,.15,.20,.12,.08]),
    'device'      : rng.choice(['Mobile','Desktop','TV','Tablet'], N_USERS,
                                p=[.55,.25,.12,.08]),
    'account_type': rng.choice(['free','premium'], N_USERS, p=[.75,.25]),
    'engagement'  : engagement,
    # Pre-experiment avg daily watch time: correlated with engagement via shared latent factor
    'pre_watch_min': np.exp(rng.normal(3.10 + 0.35 * engagement, 0.50, N_USERS)),
})

n_ctrl = (users['variant'] == 'control').sum()
n_trtm = (users['variant'] == 'treatment').sum()

# Churn assignment
is_ctrl = (users['variant'] == 'control').values
churn_flag = np.zeros(N_USERS, dtype=bool)
churn_flag[ is_ctrl] = rng.random(n_ctrl) < 0.45
churn_flag[~is_ctrl] = rng.random(n_trtm) < 0.39
users['churned'] = churn_flag

print(f"Users: {N_USERS:,}  |  Control: {n_ctrl:,}  |  Treatment: {n_trtm:,}")
print(f"Churn rate, ctrl: {churn_flag[is_ctrl].mean():.1%}  |  trtm: {churn_flag[~is_ctrl].mean():.1%}")
users[['age_group','country','device','account_type','variant']].head(5)


In [ ]:
# Build sessions (user × day grid)
user_day = (
    pd.DataFrame({'user_id': np.repeat(users['user_id'].values, N_DAYS),
                  'day_idx': np.tile(np.arange(N_DAYS), N_USERS)})
    .merge(users[['user_id','variant','engagement','churned']], on='user_id')
)
user_day['date'] = EXP_START + pd.to_timedelta(user_day['day_idx'], unit='D')

p_active = np.clip(0.45 + 0.06 * user_day['engagement'].clip(-1.5, 1.5), 0.12, 0.82)
is_trtm_ud    = (user_day['variant'] == 'treatment').values
is_post_churn = user_day['churned'].values & (user_day['day_idx'].values >= 14)
p_session     = np.where(is_post_churn, 0.06,
                         np.where(is_trtm_ud, p_active * 1.03, p_active))

sessions = user_day[rng.random(len(user_day)) < p_session].copy().reset_index(drop=True)
sessions['session_id'] = np.arange(1, len(sessions) + 1)

# Session watch time: log-normal, user engagement shifts the mean
# Treatment: +2.5% on geometric mean; slight novelty boost (+1.5%) in week 1
user_base = 3.20 + 0.25 * sessions['engagement'].values
is_trtm_s = (sessions['variant'] == 'treatment').values
is_week1  = sessions['day_idx'].values < 7
log_mu = (user_base
          + np.where(is_trtm_s, 0.025, 0.0)
          + np.where(is_trtm_s & is_week1, 0.015, 0.0))
sessions['session_watch_min'] = np.exp(rng.normal(log_mu, 0.75, len(sessions))).clip(0.5, 240)

vid_mu = np.where(is_trtm_s, 6.2, 7.8)
sessions['n_videos'] = np.clip(rng.normal(vid_mu, 2.5, len(sessions)), 1, 25).astype(int)

print(f"Sessions generated: {len(sessions):,}")
sessions[['session_id','user_id','date','variant','session_watch_min','n_videos']].head(5)


In [ ]:
# Video interactions (expand sessions by n_videos)
VTYPES = ['short_entertainment','long_entertainment','long_educational','short_educational']
P_CTRL = [.50, .30, .12, .08]
P_TRTM = [.30, .28, .32, .10]    # algorithm shifts mix toward long educational

sx = sessions.loc[sessions.index.repeat(sessions['n_videos'])].reset_index(drop=True)
is_t = (sx['variant'] == 'treatment').values
vtype = np.empty(len(sx), dtype=object)
vtype[~is_t] = rng.choice(VTYPES, (~is_t).sum(), p=P_CTRL)
vtype[ is_t] = rng.choice(VTYPES,   is_t.sum(),  p=P_TRTM)
sx['video_type'] = vtype

vlen = np.select(
    [sx['video_type']=='short_entertainment', sx['video_type']=='short_educational',
     sx['video_type']=='long_entertainment'],
    [rng.choice([30,60,90,120,180], len(sx)),
     rng.choice([60,120,180,300], len(sx)),
     rng.integers(300, 1200, len(sx))],
    default=rng.integers(600, 3600, len(sx))
)
sx['video_length_sec'] = vlen

a = np.select([sx['video_type']=='short_entertainment', sx['video_type']=='short_educational',
               sx['video_type']=='long_entertainment'],  [3.0,2.5,2.0], default=2.0)
b = np.select([sx['video_type']=='short_entertainment', sx['video_type']=='short_educational',
               sx['video_type']=='long_entertainment'],  [2.0,2.0,3.0], default=2.5)
sx['completion_rate'] = rng.beta(a, b).round(3)
sx['watch_time_sec']  = (sx['video_length_sec'] * sx['completion_rate']).round(1)

is_long = sx['video_type'].str.startswith('long')
is_edu  = sx['video_type'].str.contains('educational')
sx['liked']     = (rng.random(len(sx)) < np.where(is_long, .12, .08)).astype(int)
sx['shared']    = (rng.random(len(sx)) < np.where(is_long, .04, .02)).astype(int)
sx['commented'] = (rng.random(len(sx)) < np.where(is_edu, .025, .015)).astype(int)
sx['skipped']   = (sx['completion_rate'] < .10).astype(int)
sx['interaction_id'] = np.arange(1, len(sx) + 1)
interactions = sx[['interaction_id','session_id','user_id','date','day_idx','variant',
                    'video_type','video_length_sec','watch_time_sec','completion_rate',
                    'liked','shared','commented','skipped']].copy()

print(f"Video interactions: {len(interactions):,}")
print(f"\nDataset summary:")
print(f"  Users          : {N_USERS:,}")
print(f"  Sessions       : {len(sessions):,}")
print(f"  Interactions   : {len(interactions):,}")
print(f"  Date range     : {EXP_START.date()} → {(EXP_START + pd.Timedelta(days=N_DAYS-1)).date()}")


---
## Section 2: Sample Ratio Mismatch (SRM) Check

Before any analysis, I verify the experiment was correctly split.
A significant SRM (p < 0.05) would indicate a bug in the randomization
or logging pipeline, invalidating all downstream results.


In [ ]:
chi2_srm, p_srm = stats.chisquare([n_ctrl, n_trtm], f_exp=[N_USERS/2, N_USERS/2])
print(f"Expected  : {N_USERS//2:,} per variant")
print(f"Observed  : control={n_ctrl:,}  treatment={n_trtm:,}")
print(f"χ² = {chi2_srm:.3f}   p = {p_srm:.4f}")
print()
if p_srm > 0.05:
    print("✅ No SRM detected. Randomization appears healthy, proceeding with analysis.")
else:
    print("⚠️  SRM detected. Investigate randomization pipeline before proceeding.")


---
## Section 3: KPI Framework

Before running any tests I define the success criteria explicitly.
This prevents p-hacking and keeps the analysis anchored to business outcomes.

| Tier | Metric | Definition | Direction |
|---|---|---|---|
| **Primary** | Avg daily watch time / user | Total watch time ÷ 30 days | ↑ |
| **Secondary** | Avg session duration | Mean watch time per session | ↑ |
| **Secondary** | Week-4 retention | % of week-1 users active in week 4 | ↑ |
| **Secondary** | Videos per session | Avg # of videos per session | monitor |
| **Guardrail** | Short-form impressions share | % of views going to short-form content | ↓ flagged if < −5 pp |
| **Guardrail** | Entertainment creator views | % of views going to entertainment content | ↓ flagged if < −5 pp |

**Decision rule:** Ship if primary metric is significant and positive, AND no guardrail metric
falls outside its acceptable range. Secondary metrics inform *why* the lift happened.


---
## Section 4: Primary Metric Analysis

In [ ]:
# Aggregate total watch time per user; zero-fill users with no sessions
user_total = (sessions
    .groupby(['user_id','variant'])['session_watch_min']
    .sum().reset_index(name='total_watch_min'))
user_total['avg_daily_watch_min'] = user_total['total_watch_min'] / N_DAYS

# Merge back to full user list so non-active users get 0
all_uw = (users[['user_id','variant','pre_watch_min']]
    .merge(user_total[['user_id','avg_daily_watch_min']], on='user_id', how='left')
    .fillna({'avg_daily_watch_min': 0}))

ctrl_w = all_uw.loc[all_uw['variant']=='control',   'avg_daily_watch_min']
trtm_w = all_uw.loc[all_uw['variant']=='treatment', 'avg_daily_watch_min']

print("Primary metric, avg daily watch time per user:")
print(f"  Control   : mean={ctrl_w.mean():.2f}  median={ctrl_w.median():.2f}  σ={ctrl_w.std():.2f}")
print(f"  Treatment : mean={trtm_w.mean():.2f}  median={trtm_w.median():.2f}  σ={trtm_w.std():.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for v, color in [('control', BLUE), ('treatment', RED)]:
    vals = all_uw.loc[all_uw['variant']==v, 'avg_daily_watch_min']
    ax1.hist(vals.clip(0, vals.quantile(.99)), bins=55, alpha=0.55,
             color=color, density=True, label=f"{v.title()} (μ={vals.mean():.1f} min)")
    ax1.axvline(vals.mean(), color=color, lw=2, ls='--')
ax1.set_xlabel('Avg Daily Watch Time (min/day)')
ax1.set_ylabel('Density')
ax1.set_title('Distribution, Avg Daily Watch Time')
ax1.legend()

# Daily time-series (avg per active session per day)
udw = (sessions.groupby(['user_id','date','day_idx','variant'])['session_watch_min']
       .sum().reset_index())
ts = udw.groupby(['date','variant'])['session_watch_min'].mean().reset_index()
for v, color in [('control',BLUE),('treatment',RED)]:
    sub = ts[ts['variant']==v].sort_values('date')
    smooth = sub['session_watch_min'].rolling(3, min_periods=1).mean()
    ax2.plot(sub['date'], smooth, color=color, lw=2.2, label=v.title())
    ax2.fill_between(sub['date'], smooth, alpha=0.12, color=color)
ax2.set_xlabel('Date')
ax2.set_ylabel('Avg Watch / Session (min)')
ax2.set_title('Daily Session Watch Time Trend (3-day rolling)')
ax2.legend()
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Statistical test: Welch's two-sample t-test (unequal variance)
t_stat, p_val  = stats.ttest_ind(trtm_w, ctrl_w, equal_var=False)
delta_abs = trtm_w.mean() - ctrl_w.mean()
delta_rel = delta_abs / ctrl_w.mean()
cohens_d  = delta_abs / np.sqrt((trtm_w.std()**2 + ctrl_w.std()**2) / 2)

# Bootstrap 95% CI for the mean difference (5,000 resamples)
def boot_ci(a, b, n_boot=5_000, alpha=0.05):
    diffs = [rng.choice(a, len(a), replace=True).mean() -
             rng.choice(b, len(b), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(diffs, [alpha/2*100, (1-alpha/2)*100])

ci_lo, ci_hi = boot_ci(trtm_w.values, ctrl_w.values)

print("=" * 55)
print("  PRIMARY METRIC STATISTICAL RESULTS")
print("=" * 55)
print(f"  Δ (absolute)   : {delta_abs:+.3f} min/day")
print(f"  Δ (relative)   : {delta_rel:+.1%}")
print(f"  t-statistic    : {t_stat:.3f}")
print(f"  p-value        : {p_val:.4e}")
print(f"  Cohen's d      : {cohens_d:.3f}  (small=0.2, medium=0.5)")
print(f"  Bootstrap 95% CI : [{ci_lo:.3f}, {ci_hi:.3f}] min/day")
print("=" * 55)
decision = "✅ REJECT H₀, effect is statistically significant" if p_val < 0.05 else "❌ FAIL TO REJECT H₀"
print(f"  Decision (α=0.05) : {decision}")


### CUPED: Controlled-experiment Using Pre-Experiment Data

CUPED adjusts each user's in-experiment watch time by their pre-experiment baseline,
removing variance that's unrelated to the treatment. This reduces the standard error
of the treatment effect estimate, a standard technique at companies like Google, Netflix,
and Airbnb for making experiments more sensitive.

**Adjusted metric:**
`Y_cuped = Y - θ × (X_pre - E[X_pre])`

where `θ = Cov(Y, X_pre) / Var(X_pre)` and `X_pre` is pre-experiment avg daily watch time.


In [ ]:
theta = (np.cov(all_uw['avg_daily_watch_min'], all_uw['pre_watch_min'])[0,1] /
         all_uw['pre_watch_min'].var())

all_uw['cuped_watch'] = (all_uw['avg_daily_watch_min']
    - theta * (all_uw['pre_watch_min'] - all_uw['pre_watch_min'].mean()))

cuped_ctrl = all_uw.loc[all_uw['variant']=='control',   'cuped_watch']
cuped_trtm = all_uw.loc[all_uw['variant']=='treatment', 'cuped_watch']
_, p_cuped   = stats.ttest_ind(cuped_trtm, cuped_ctrl, equal_var=False)
var_red      = 1 - cuped_trtm.var() / trtm_w.var()
cuped_delta  = cuped_trtm.mean() - cuped_ctrl.mean()

print(f"  θ (theta)           : {theta:.4f}")
print(f"  Variance reduction  : {var_red:.1%}")
print(f"  CUPED Δ             : {cuped_delta:+.3f} min/day")
print(f"  CUPED p-value       : {p_cuped:.4e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axes,
    ['avg_daily_watch_min','cuped_watch'],
    ['Raw Metric', 'CUPED-Adjusted Metric']):
    for v, color in [('control',BLUE),('treatment',RED)]:
        vals = all_uw.loc[all_uw['variant']==v, col].clip(-5, 60)
        ax.hist(vals, bins=55, alpha=0.55, color=color,
                label=f"{v.title()} σ={all_uw.loc[all_uw['variant']==v,col].std():.2f}", density=True)
        ax.axvline(vals.mean(), color=color, lw=2, ls='--')
    ax.set_title(title)
    ax.set_xlabel('Watch Time (min/day)')
    ax.legend(fontsize=9)
plt.suptitle(f'CUPED Variance Reduction: {var_red:.1%}  (θ = {theta:.4f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 5: Secondary Metrics

In [ ]:
sc = sessions[sessions['variant']=='control']
st = sessions[sessions['variant']=='treatment']

# Session duration
t_dur, p_dur = stats.ttest_ind(st['session_watch_min'], sc['session_watch_min'], equal_var=False)
delta_dur = st['session_watch_min'].mean() - sc['session_watch_min'].mean()

# Videos per session
t_vid, p_vid = stats.ttest_ind(st['n_videos'], sc['n_videos'], equal_var=False)
delta_vid = st['n_videos'].mean() - sc['n_videos'].mean()

# Week-4 retention
w1_users = set(sessions[sessions['day_idx'] < 7]['user_id'].unique())
w4_users = set(sessions[sessions['day_idx'] >= 21]['user_id'].unique())
ret_df = users[users['user_id'].isin(w1_users)][['user_id','variant']].copy()
ret_df['retained'] = ret_df['user_id'].isin(w4_users).astype(int)
ret_ctrl = ret_df[ret_df['variant']=='control']['retained'].mean()
ret_trtm = ret_df[ret_df['variant']=='treatment']['retained'].mean()
ct = pd.crosstab(ret_df['variant'], ret_df['retained'])
chi2_ret, p_ret, _, _ = stats.chi2_contingency(ct)

print("Secondary Metrics Summary:")
print(f"  Session Duration , ctrl={sc['session_watch_min'].mean():.2f} min  "
      f"trtm={st['session_watch_min'].mean():.2f} min  Δ={delta_dur:+.2f}  p={p_dur:.4f}")
print(f"  Videos/Session   , ctrl={sc['n_videos'].mean():.2f}  "
      f"trtm={st['n_videos'].mean():.2f}  Δ={delta_vid:+.2f}  p={p_vid:.4f}")
print(f"  Week-4 Retention , ctrl={ret_ctrl:.1%}  trtm={ret_trtm:.1%}  "
      f"Δ={ret_trtm-ret_ctrl:+.1%}  p={p_ret:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Session duration
ax = axes[0]
for v, color in [('control',BLUE),('treatment',RED)]:
    vals = sessions.loc[sessions['variant']==v,'session_watch_min'].clip(0, 120)
    ax.hist(vals, bins=50, alpha=0.55, color=color,
            label=f"{v.title()} μ={vals.mean():.1f}", density=True)
    ax.axvline(vals.mean(), color=color, lw=2, ls='--')
ax.set_title('Session Watch Time')
ax.set_xlabel('Session Duration (min)')
ax.legend()

# Videos per session
ax = axes[1]
for v, color in [('control',BLUE),('treatment',RED)]:
    vals = sessions.loc[sessions['variant']==v,'n_videos']
    ax.hist(vals, bins=20, alpha=0.55, color=color,
            label=f"{v.title()} μ={vals.mean():.1f}", density=True)
    ax.axvline(vals.mean(), color=color, lw=2, ls='--')
ax.set_title('Videos per Session')
ax.set_xlabel('# Videos')
ax.legend()

# Week-4 retention bars
ax = axes[2]
bars = ax.bar(['Control','Treatment'], [ret_ctrl, ret_trtm],
               color=[BLUE, RED], alpha=0.85, width=0.45)
for bar, val in zip(bars, [ret_ctrl, ret_trtm]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.007,
            f'{val:.1%}', ha='center', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_title('Week-4 Retention Rate')
ax.set_ylabel('Retention %')
ax.annotate(f'Δ = {ret_trtm-ret_ctrl:+.1%}  (p = {p_ret:.3f})',
            xy=(0.5,0.05), xycoords='axes fraction', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.show()


---
## Section 6: Guardrail Metrics: Creator Ecosystem

The new algorithm changes *which* content gets recommended. A critical guardrail:
short-form and entertainment creators should not see a significant drop in impressions.
A meaningful decline hurts creator monetization and could damage the YouTube ecosystem.


In [ ]:
type_shares = (interactions
    .groupby(['variant','video_type'])['interaction_id']
    .count().rename('count').reset_index())
type_shares['share'] = type_shares.groupby('variant')['count'].transform(lambda x: x/x.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bars
VTYPES = ['short_entertainment','long_entertainment','long_educational','short_educational']
x = np.arange(len(VTYPES))
w = 0.38
ts_piv = type_shares.pivot(index='variant', columns='video_type', values='share').fillna(0)
ax = axes[0]
for i, (v, color) in enumerate([('control',BLUE),('treatment',RED)]):
    vals = [ts_piv.loc[v, vt] if vt in ts_piv.columns else 0 for vt in VTYPES]
    ax.bar(x + i*w, vals, w, label=v.title(), color=color, alpha=0.82)
ax.set_xticks(x + w/2)
ax.set_xticklabels([t.replace('_','\n') for t in VTYPES], fontsize=9)
ax.set_title('Content-Type Impression Share')
ax.set_ylabel('Share of Impressions')
ax.legend()

# Delta chart
ax = axes[1]
deltas = [(ts_piv.loc['treatment', vt] - ts_piv.loc['control', vt]
           if vt in ts_piv.columns else 0) for vt in VTYPES]
clrs = [RED if d < -0.05 else (GREEN if d > 0.05 else AMBER) for d in deltas]
bars = ax.bar([t.replace('_','\n') for t in VTYPES], deltas, color=clrs, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(-0.05, color=RED, lw=1, ls='--', label='−5 pp guardrail threshold')
ax.set_title('Δ Impression Share (Treatment − Control)')
ax.set_ylabel('Δ Share')
ax.legend(fontsize=9)
for bar, val in zip(bars, deltas):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:+.1%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("⚠️  Guardrail flags:")
for vt, d in zip(VTYPES, deltas):
    flag = '⚠️  FLAGGED' if d < -0.05 else '✅ OK'
    print(f"  {vt:28s}  Δ={d:+.1%}  {flag}")


---
## Section 7: Segmentation Analysis

Segment-level analysis serves two purposes:
1. **Identify where the algorithm works best** (prioritize expansion / messaging)
2. **Detect heterogeneous harm** (guardrail: no segment should be significantly harmed)


In [ ]:
full = all_uw.merge(users[['user_id','device','age_group','account_type']], on='user_id')
seg_results = {}

for seg in ['device','age_group','account_type']:
    rows = []
    for val in sorted(full[seg].unique()):
        sub = full[full[seg]==val]
        c_  = sub.loc[sub['variant']=='control',   'avg_daily_watch_min']
        t_  = sub.loc[sub['variant']=='treatment', 'avg_daily_watch_min']
        if len(c_) < 10 or len(t_) < 10:
            continue
        d_  = t_.mean() - c_.mean()
        _, p_ = stats.ttest_ind(t_, c_, equal_var=False)
        rows.append({'segment':val,'n':len(sub),'ctrl':c_.mean(),'trtm':t_.mean(),
                     'delta':d_,'pct_lift':d_/c_.mean(),'pvalue':p_})
    seg_results[seg] = pd.DataFrame(rows).sort_values('delta', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, seg in zip(axes, ['device','age_group','account_type']):
    df = seg_results[seg].sort_values('delta')
    clrs = [GREEN if d > 0 else '#D32F2F' for d in df['delta']]
    bars = ax.barh(df['segment'], df['delta'], color=clrs, alpha=0.82, height=0.55)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(delta_abs, color='gray', lw=1, ls='--', label=f'Overall Δ')
    ax.set_title(f'Lift by {seg.replace("_"," ").title()}')
    ax.set_xlabel('Δ Avg Daily Watch (min/day)')
    ax.legend(fontsize=8)
    for bar, val, p_ in zip(bars, df['delta'], df['pvalue']):
        sig = ' ✱' if p_ < 0.05 else ''
        ax.text(bar.get_width() + (0.03 if val >= 0 else -0.03),
                bar.get_y()+bar.get_height()/2, f'{val:+.2f}{sig}', va='center', fontsize=9)
plt.suptitle('Segment-Level Treatment Effect', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

for seg, df in seg_results.items():
    print(f"\n{seg.upper()}")
    print(df[['segment','ctrl','trtm','delta','pct_lift','pvalue']].to_string(index=False))


---
## Section 8: Novelty Effect Check

A classic pitfall in recommendation experiments: users engage more with anything "new"
in week 1 (novelty effect), then revert to baseline. A treatment effect that shrinks
over time may not reflect genuine long-term value.

I measure weekly treatment lifts to distinguish **novelty** (lift decreases over time)
from **genuine engagement change** (lift is stable or grows).


In [ ]:
udw2 = (sessions.groupby(['user_id','date','day_idx','variant'])['session_watch_min']
        .sum().reset_index())

weekly_lift = []
for wk, (lo, hi) in enumerate([(0,6),(7,13),(14,20),(21,29)], 1):
    sub = udw2[udw2['day_idx'].between(lo, hi)]
    uw  = sub.groupby(['user_id','variant'])['session_watch_min'].sum().reset_index()
    n_days_w = hi - lo + 1
    uw['daily'] = uw['session_watch_min'] / n_days_w
    uw_f = (users[['user_id','variant']]
            .merge(uw[['user_id','daily']], on='user_id', how='left')
            .fillna({'daily': 0}))
    c_ = uw_f.loc[uw_f['variant']=='control',   'daily']
    t_ = uw_f.loc[uw_f['variant']=='treatment', 'daily']
    d_ = t_.mean() - c_.mean()
    _, p_ = stats.ttest_ind(t_, c_, equal_var=False)
    weekly_lift.append({'week':f'Week {wk}','ctrl':c_.mean(),'trtm':t_.mean(),'delta':d_,'pvalue':p_})

weekly_df = pd.DataFrame(weekly_lift)
w1_lift   = weekly_df.loc[0,'delta']
w4_lift   = weekly_df.loc[3,'delta']

fig, ax = plt.subplots(figsize=(8, 5))
clrs = [GREEN if d > 0 else RED for d in weekly_df['delta']]
ax.bar(weekly_df['week'], weekly_df['delta'], color=clrs, alpha=0.82, width=0.55)
ax.axhline(delta_abs, color='black', lw=1.5, ls='--', label=f'Overall Δ = {delta_abs:.2f} min/day')
ax.set_ylabel('Δ Avg Daily Watch (min/day)')
ax.set_title('Weekly Treatment Effect, Novelty Check')
ax.legend(fontsize=10)
for i, (d, p_) in enumerate(zip(weekly_df['delta'], weekly_df['pvalue'])):
    lbl = '***' if p_ < 0.001 else ('*' if p_ < 0.05 else 'n.s.')
    ax.text(i, d + 0.01, lbl, ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Week-1 lift : {w1_lift:+.3f} min/day")
print(f"Week-4 lift : {w4_lift:+.3f} min/day")
if w1_lift > w4_lift * 1.30:
    print("⚠️  Possible novelty effect: week-1 lift is substantially larger than week-4.")
    print("    Recommend extending experiment or monitoring for reversion.")
else:
    print("✅ No novelty effect. Lift is durable across all four weeks.")


---
## Section 9: Root Cause Decomposition

The primary metric gain (+~1.5 min/day) can come from two levers:
1. **More sessions**: users are visiting YouTube more frequently
2. **Longer sessions**: users are staying longer each time they visit

I decompose the lift using the standard product analytics framework:
`Watch/user/day = Sessions/user/day × Avg_session_watch`

This helps product and engineering teams know *where to focus* if they want to
amplify or protect the effect.


In [ ]:
# Per-user-per-day session frequency (unconditional: includes zero-session days)
ctrl_freq = sessions[sessions['variant']=='control']['user_id'].count() / n_ctrl / N_DAYS
trtm_freq = sessions[sessions['variant']=='treatment']['user_id'].count() / n_trtm / N_DAYS

ctrl_dur = sessions[sessions['variant']=='control']['session_watch_min'].mean()
trtm_dur = sessions[sessions['variant']=='treatment']['session_watch_min'].mean()

baseline          = ctrl_freq * ctrl_dur
freq_contribution = (trtm_freq - ctrl_freq) * ctrl_dur
dur_contribution  = ctrl_freq * (trtm_dur - ctrl_dur)
interaction_term  = (trtm_freq - ctrl_freq) * (trtm_dur - ctrl_dur)
total_lift        = freq_contribution + dur_contribution + interaction_term

print("Root Cause Decomposition")
print(f"  Session frequency  : ctrl={ctrl_freq:.4f}/day  trtm={trtm_freq:.4f}/day  "
      f"Δ={trtm_freq-ctrl_freq:+.4f}")
print(f"  Session duration   : ctrl={ctrl_dur:.2f} min   trtm={trtm_dur:.2f} min   "
      f"Δ={trtm_dur-ctrl_dur:+.2f}")
print()
print(f"  Lift decomposition (min/day):")
print(f"    Frequency effect : {freq_contribution:+.3f}  ({freq_contribution/total_lift:.0%} of lift)")
print(f"    Duration effect  : {dur_contribution:+.3f}  ({dur_contribution/total_lift:.0%} of lift)")
print(f"    Interaction      : {interaction_term:+.3f}  ({interaction_term/total_lift:.0%} of lift)")
print(f"    Total            : {total_lift:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Decomposition waterfall
ax = axes[0]
components = ['Frequency\nEffect', 'Duration\nEffect', 'Interaction']
values = [freq_contribution, dur_contribution, interaction_term]
clrs2 = [GREEN if v >= 0 else RED for v in values]
bars = ax.bar(components, values, color=clrs2, alpha=0.85, width=0.5)
ax.axhline(0, color='black', lw=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + (0.001 if val >= 0 else -0.004),
            f'{val:+.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Watch-Time Lift Decomposition')
ax.set_ylabel('Δ Watch Time (min/day per user)')

# Pie chart for share
ax = axes[1]
labels = ['More sessions\n(frequency)', 'Longer sessions\n(duration)', 'Interaction']
sizes  = [abs(v) for v in values]
pie_clrs = [GREEN, BLUE, AMBER]
ax.pie(sizes, labels=labels, colors=pie_clrs, autopct='%1.0f%%', startangle=90,
       textprops={'fontsize': 10})
ax.set_title('Share of Lift by Driver')

plt.tight_layout()
plt.show()
print(f"\n→ Primary driver: {'session frequency (churn reduction)' if abs(freq_contribution) > abs(dur_contribution) else 'session duration (longer content)'}")


---
## Section 10: Recommendation

### Summary of Findings

| Metric | Control | Treatment | Δ | Significant? |
|---|---|---|---|---|
| **Avg daily watch time** | 11.96 min | 13.49 min | **+1.53 min (+12.8%)** | ✅ p < 0.001 |
| Session duration | 33.7 min | 35.3 min | +1.6 min (+4.8%) | ✅ p < 0.001 |
| Videos / session | 7.33 | 5.71 | −1.62 (−22%) | ✅ p < 0.001 |
| Week-4 retention | 73.9% | 77.8% | **+3.9 pp** | ✅ p = 0.001 |
| Short-form impressions | 50.2% | 29.8% | **−20.4 pp** | ⚠️ GUARDRAIL |
| Entertainment impressions | 29.8% | 28.2% | −1.6 pp | ✅ OK |

---

### Recommendation: **Conditional launch: fix the guardrail before full rollout**

The treatment effect is real and durable. The new algorithm meaningfully increases
watch time (+12.8%) and long-run retention (+3.9 pp), which are the metrics that drive
creator ad revenue and platform health. The root cause is a combination of users having
slightly more frequent sessions (lower churn) and staying longer per session.

**However, there is a clear guardrail violation: short-entertainment creators are
receiving ~20 percentage points fewer impressions.** This is a large shift and likely
to hurt this creator segment's engagement and earnings.

**Recommended next steps:**

1. **Do not ship at current settings.** The short-form creator impact is too large to ignore.
2. **Adjust the recommendation blend.** Explore a parameter that limits how aggressively
   educational content is surfaced, target a max 10 pp shift in short-form impressions.
3. **Re-run the experiment** with the blended algorithm. If the guardrail passes and
   primary metric remains positive, ship with creator ecosystem monitoring in place.
4. **Monitor week-over-week** watch time stability post-launch. The novelty check is
   clean, but the first 2 weeks of production data will confirm durability at scale.

---

*Analysis by Praneeth Korukonda | Dataset: synthetically generated for methodological demonstration*
